# 02. Uplift Modeling (X-Learner)

**Цель**  
Оценить индивидуальный прирост конверсии (CATE) от маркетингового предложения.

**Метод**: X-Learner — один из самых стабильных meta-learners.

**Метрики**: AUUC, Qini-кривая.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from src.uplift import XLearner
from src.utils import auuc_score, qini_curve

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Загрузка подготовленных данных

In [ ]:
data = joblib.load(PROJECT_ROOT / "outputs" / "prepared_data.joblib")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
t_train = data["t_train"]
t_test = data["t_test"]
feature_names = data["feature_names"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Обучение X-Learner

In [ ]:
# Параметры можно тюнить через Optuna, здесь зафиксированы хорошие значения
clf_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 5,
    "num_leaves": 31,
    "min_child_samples": 40,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}

reg_params = clf_params.copy()
reg_params.update({
    "reg_alpha": 2.0,
    "reg_lambda": 2.0,
    "min_child_weight": 10,
})

model = XLearner(
    propensity_params=clf_params,
    outcome_params=clf_params,
    effect_params=reg_params,
    clip_range=(-0.15, 0.30),
)

model.fit(X_train, t_train, y_train)
print("X-Learner trained successfully.")

## 3. Предсказание и оценка

In [ ]:
uplift_test = model.predict_uplift(X_test)

auuc = auuc_score(y_test, uplift_test, t_test)
print(f"AUUC on test: {auuc:.5f}")
print(f"Mean uplift score: {uplift_test.mean():.4f}")
print(f"Std  uplift score: {uplift_test.std():.4f}")
print(f"Min / Max: {uplift_test.min():.4f} / {uplift_test.max():.4f}")

In [ ]:
# Qini curve
x, qini = qini_curve(y_test, uplift_test, t_test)

# Random baseline
n = len(y_test)
n_t = t_test.sum()
n_c = n - n_t
overall_uplift = (
    y_test[t_test == 1].mean() - y_test[t_test == 0].mean()
)
random_qini = overall_uplift * n_t * x

plt.figure(figsize=(9, 5))
plt.plot(x, qini, label="X-Learner", linewidth=2)
plt.plot(x, random_qini, label="Random", linestyle="--", color="gray")
plt.xlabel("Fraction of population (ranked by uplift)")
plt.ylabel("Cumulative incremental conversions (Qini)")
plt.title("Qini Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "02_qini_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Feature Importance (outcome model treated)

In [ ]:
imp = pd.DataFrame({
    "feature": feature_names,
    "importance": model.outcome_t_.feature_importances_
}).sort_values("importance", ascending=False)

print("Top-15 features:")
print(imp.head(15).to_string(index=False))

plt.figure(figsize=(8, 6))
sns_bar = imp.head(12)
plt.barh(sns_bar["feature"][::-1], sns_bar["importance"][::-1])
plt.title("Feature Importance (Outcome model — Treated)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "02_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Сохранение модели и предсказаний

In [ ]:
joblib.dump(model, PROJECT_ROOT / "models" / "xlearner.joblib")

np.save(PROJECT_ROOT / "outputs" / "uplift_test.npy", uplift_test)

results = pd.DataFrame({
    "idx": data["idx_test"],
    "y_true": y_test,
    "treatment": t_test,
    "uplift_score": uplift_test,
})
results.to_csv(PROJECT_ROOT / "outputs" / "uplift_predictions_test.csv", index=False)

print("Saved:")
print(" - models/xlearner.joblib")
print(" - outputs/uplift_test.npy")
print(" - outputs/uplift_predictions_test.csv")

## Выводы

- X-Learner успешно обучен.
- AUUC > 0 означает, что модель лучше случайного таргетинга.
- Qini-кривая показывает накопленный прирост конверсий при ранжировании по uplift.
- Далее используем `uplift_score` для построения контактной политики и оценки экономики.